In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("reddit_text_10000_labeled.csv")
data.head()

                                               title  ... num_toxic_comments
0  So what are auctioned houses going for compare...  ...                  0
1                     Hows the Architecture Industry  ...                  0
2               Small House Storage Ideas/Solutions?  ...                  0
3                      Worked at a StartUp it Sucked  ...                  0
4  [HIRING] Backend, Infra & Agentic AI Engineers...  ...                  0

[5 rows x 5 columns]

In [3]:
import json

# Post-level cutoff, distinct from the comment model's 0.5 score threshold.
# 4% gives the closest available balance on the completed 1,000-post sample.
# 558 posts have exactly 0% toxicity, so no percentage cutoff can give 50/50.
TOXIC_PERCENT_THRESHOLD = 10
if not 0 < TOXIC_PERCENT_THRESHOLD <= 100:
    raise ValueError("TOXIC_PERCENT_THRESHOLD must be in (0, 100]")

def count_comments(value):
    comments = json.loads(value)
    if not isinstance(comments, list) or not all(isinstance(text, str) for text in comments):
        raise ValueError("Expected comments to be a JSON array of strings")
    return len(comments)

data["num_comments"] = data["comments"].apply(count_comments)
toxic_counts = pd.to_numeric(data["num_toxic_comments"], errors="raise")
if (data["num_comments"] == 0).any():
    raise ValueError("Cannot calculate toxic percentage for posts with no comments")
if (toxic_counts.isna().any() or not np.isfinite(toxic_counts).all()
        or (toxic_counts < 0).any() or (toxic_counts > data["num_comments"]).any()
        or (toxic_counts % 1 != 0).any()):
    raise ValueError("Toxic comment counts must be integers between 0 and num_comments")

data["toxic_comment_percentage"] = 100 * toxic_counts / data["num_comments"]
# Apply the cutoff before any rounding for display.
data["toxic_index"] = (data["toxic_comment_percentage"] >= TOXIC_PERCENT_THRESHOLD).astype(int)
TARGET_NAMES = [f"0: below {TOXIC_PERCENT_THRESHOLD:g}% toxic",
                f"1: at least {TOXIC_PERCENT_THRESHOLD:g}% toxic"]

toxic_index_counts = data["toxic_index"].value_counts().reindex([0, 1], fill_value=0)
toxic_index_counts.index.name = "toxic_index"
toxic_index_counts.name = "number_of_posts"
print(f"Positive target: at least {TOXIC_PERCENT_THRESHOLD:g}% of comments are toxic")
print(toxic_index_counts.to_string())
print(data[["num_comments", "num_toxic_comments", "toxic_comment_percentage", "toxic_index"]].head())


Positive target: at least 10% of comments are toxic
toxic_index
0    7776
1    2224
   num_comments  num_toxic_comments  toxic_comment_percentage  toxic_index
0            10                   0                       0.0            0
1            10                   0                       0.0            0
2            16                   0                       0.0            0
3            12                   0                       0.0            0
4             6                   0                       0.0            0


## Predict whether a post attracts toxic comments with a linear SVM

Use **title + description** as input and `toxic_index` as the target (1 = the toxic-comment percentage meets or exceeds `TOXIC_PERCENT_THRESHOLD`, default 4%). Comments and toxicity scores are excluded from the input because they define the target. These are model-generated labels, not human annotations.

Hold out 20% of posts for evaluation. Learn the TF-IDF vocabulary only from training posts, then train a linear support vector machine. Exact duplicate input texts are kept in the same split to prevent leakage.

The target is `100 × num_toxic_comments / num_comments`. For example, 1 toxic comment out of 25 is 4% (positive), while 1 out of 100 is 1% (negative). This normalizes by comment volume, but percentages from a few comments are still less stable. The 4% cutoff gives 442 positive and 558 negative posts on the completed 1,000-post sample, the closest possible balance because 558 posts have zero toxic comments. On this sample, it yields the same labels as the original any-toxic-comment rule; a larger sample may differ. Keep the threshold fixed before evaluating new data. Count and percentage columns are target metadata, not model inputs. Previous model scores used a different target and must be recomputed.


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score

X = data["title"].fillna("").astype(str) + " " + data["description"].fillna("").astype(str)
y = data["toxic_index"]

# Split unique normalized texts, so duplicate posts cannot cross the split.
groups = X.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
group_labels = pd.DataFrame({"group": groups, "label": y}).groupby("group", sort=False)["label"].max()
train_groups, test_groups = train_test_split(
    group_labels.index.to_numpy(), test_size=0.2, random_state=42, stratify=group_labels
)
train_mask = groups.isin(train_groups)
test_mask = groups.isin(test_groups)
X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
assert set(groups.loc[train_mask]).isdisjoint(groups.loc[test_mask])
print(f"Training posts: {len(X_train)}; test posts: {len(X_test)}")
print(pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()}).sort_index())


Training posts: 7999; test posts: 2001
             train  test
toxic_index             
0             6219  1557
1             1780   444


## Tokenization and TF-IDF

The vectorizer lowercases text and splits it into word tokens of at least two characters, ignoring punctuation. It uses single words (unigrams) and adjacent word pairs (bigrams). TF-IDF weights terms by their frequency within a post and their rarity across training posts. The SVM receives these numeric features, not raw text.


In [5]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True, ngram_range=(1, 2), min_df=2,
        max_features=30000, sublinear_tf=True
    )),
    ("svm", LinearSVC(C=1.0, class_weight="balanced", random_state=42, max_iter=10000)),
])

vectorizer = svm_pipeline.named_steps["tfidf"]
example_text = X_train.iloc[0]
tokens = vectorizer.build_tokenizer()(vectorizer.build_preprocessor()(example_text))
print("Text preview:", example_text[:400])
print("First 30 word tokens:", tokens[:30])
print("First 40 unigram/bigram features:", vectorizer.build_analyzer()(example_text)[:40])


Text preview: So what are auctioned houses going for compared to their ratable values right now? Thinking about the Waikato. We are in the market for a home and I keep seeing auctions pop up. So are they worth going for? Just wondering your general opinion and what they're going for on average compared to their ratable values.
First 30 word tokens: ['so', 'what', 'are', 'auctioned', 'houses', 'going', 'for', 'compared', 'to', 'their', 'ratable', 'values', 'right', 'now', 'thinking', 'about', 'the', 'waikato', 'we', 'are', 'in', 'the', 'market', 'for', 'home', 'and', 'keep', 'seeing', 'auctions', 'pop']
First 40 unigram/bigram features: ['so', 'what', 'are', 'auctioned', 'houses', 'going', 'for', 'compared', 'to', 'their', 'ratable', 'values', 'right', 'now', 'thinking', 'about', 'the', 'waikato', 'we', 'are', 'in', 'the', 'market', 'for', 'home', 'and', 'keep', 'seeing', 'auctions', 'pop', 'up', 'so', 'are', 'they', 'worth', 'going', 'for', 'just', 'wondering', 'your']


In [6]:
# fit learns both TF-IDF weights and the SVM from training data only.
svm_pipeline.fit(X_train, y_train)

example_vector = vectorizer.transform([example_text])
feature_names = vectorizer.get_feature_names_out()
example_features = pd.DataFrame({
    "feature": feature_names[example_vector.indices],
    "tfidf_weight": example_vector.data,
}).sort_values("tfidf_weight", ascending=False)
print(f"Training feature matrix shape: ({len(X_train)}, {len(feature_names)})")
print("Highest-weight features for the example post:")
print(example_features.head(15).to_string(index=False))


Training feature matrix shape: (7999, 30000)
Highest-weight features for the example post:
    feature  tfidf_weight
  going for      0.331175
     values      0.238650
   to their      0.214507
compared to      0.202573
   compared      0.197463
     so are      0.182222
opinion and      0.177898
     pop up      0.174243
keep seeing      0.161462
 market for      0.160494
   for home      0.160494
 on average      0.156171
   for just      0.155393
      up so      0.155393
        pop      0.149949


## Tune the SVM using training data only

Compare 32 configurations with three-fold stratified group cross-validation. Duplicate normalized texts stay together in every fold. Each fold learns its own vocabulary and TF-IDF weights. Search word unigrams versus unigrams + bigrams, minimum document frequency, SVM regularization (`C`), and class weighting. Select by mean toxic-class F1; also report balanced accuracy. The original SVM settings are included in the search.

One worker limits competition with the running toxicity-labeling job. Temporary caching reuses TF-IDF transformations across SVM settings and is removed after fitting. The held-out test set is used only for the final comparison; tuning does not guarantee a higher test score.


In [7]:
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import make_scorer
from joblib import Memory
from tempfile import TemporaryDirectory

cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
train_group_ids = groups.loc[X_train.index].to_numpy()
cv_splits = list(cv.split(X_train, y_train, groups=train_group_ids))
for fit_indices, validation_indices in cv_splits:
    assert set(train_group_ids[fit_indices]).isdisjoint(train_group_ids[validation_indices])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "svm__C": [0.1, 1.0, 10.0, 0.01],
    "svm__class_weight": [None, "balanced"],
}
# 2 x 2 x 4 x 2 = 32 candidates, each evaluated on 3 training folds.
with TemporaryDirectory(prefix="reddit_svm_cache_") as cache_dir:
    candidate = clone(svm_pipeline).set_params(memory=Memory(cache_dir, verbose=0))
    search = GridSearchCV(
        candidate, param_grid=param_grid, cv=cv_splits,
        scoring={"toxic_f1": make_scorer(f1_score, zero_division=0),
                 "balanced_accuracy": "balanced_accuracy"},
        refit="toxic_f1", n_jobs=1, verbose=1, error_score="raise",
    )
    search.fit(X_train, y_train)
    tuned_svm = search.best_estimator_.set_params(memory=None)

print("Selected parameters:", search.best_params_)
print(f"Mean cross-validation toxic F1: {search.best_score_:.3f}")
cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_toxic_f1")
print(cv_results[["mean_test_toxic_f1", "std_test_toxic_f1",
                  "mean_test_balanced_accuracy", "params"]].head(5).to_string(index=False))


Fitting 3 folds for each of 32 candidates, totalling 96 fits
Selected parameters: {'svm__C': 0.01, 'svm__class_weight': 'balanced', 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
Mean cross-validation toxic F1: 0.520
 mean_test_toxic_f1  std_test_toxic_f1  mean_test_balanced_accuracy                                                                                              params
           0.520334           0.013889                     0.706279 {'svm__C': 0.01, 'svm__class_weight': 'balanced', 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
           0.518908           0.008780                     0.700369  {'svm__C': 0.1, 'svm__class_weight': 'balanced', 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
           0.518598           0.014376                     0.701844 {'svm__C': 0.01, 'svm__class_weight': 'balanced', 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
           0.518557           0.013032                     0.705032 {'svm__C': 0.01, 'svm__class_weight': 'b

## Evaluate on held-out posts

Compare against a baseline that always predicts the most common training class. For class 1, precision measures how often a predicted toxic-comment post is correct; recall measures how many actual toxic-comment posts are found. The confusion matrix has actual classes in rows and predicted classes in columns. This measures agreement with the toxicity model's labels, not verified toxicity or causation.


In [8]:
original_pred = svm_pipeline.predict(X_test)
y_pred = tuned_svm.predict(X_test)
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(np.zeros((len(y_train), 1)), y_train)
baseline_pred = baseline.predict(np.zeros((len(y_test), 1)))

metrics = pd.DataFrame([
    {"model": name, "accuracy": accuracy_score(y_test, predictions),
     "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
     "toxic_f1": f1_score(y_test, predictions, zero_division=0)}
    for name, predictions in [("Majority baseline", baseline_pred), ("Original SVM", original_pred), ("Tuned SVM", y_pred)]
])
print(metrics.to_string(index=False))
print("\nSVM classification report:")
print(classification_report(y_test, y_pred, labels=[0, 1],
    target_names=TARGET_NAMES, zero_division=0))
confusion = pd.DataFrame(confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=["actual 0", "actual 1"], columns=["predicted 0", "predicted 1"])
print("Confusion matrix:")
print(confusion)


            model  accuracy  balanced_accuracy  toxic_f1
Majority baseline  0.778111           0.500000  0.000000
     Original SVM  0.780110           0.684824  0.508929
        Tuned SVM  0.731134           0.740292  0.555372

SVM classification report:
                       precision    recall  f1-score   support

   0: below 10% toxic       0.91      0.72      0.81      1557
1: at least 10% toxic       0.44      0.76      0.56       444

             accuracy                           0.73      2001
            macro avg       0.68      0.74      0.68      2001
         weighted avg       0.81      0.73      0.75      2001

Confusion matrix:
          predicted 0  predicted 1
actual 0         1127          430
actual 1          108          336


In [9]:
# Inspect held-out predictions. Decision scores are margins, not probabilities.
test_predictions = data.loc[X_test.index, ["title", "toxic_index"]].copy()
test_predictions["predicted_toxic_index"] = y_pred
test_predictions["svm_decision_score"] = tuned_svm.decision_function(X_test)
test_predictions.head(10)


                                                title  ...  svm_decision_score
3                       Worked at a StartUp it Sucked  ...           -0.019213
4   [HIRING] Backend, Infra & Agentic AI Engineers...  ...           -0.159562
5                    È stato bello conoscervi poverih  ...            0.088747
11  Loren Taylor gave up a career in management co...  ...            0.057069
19  Geary/19th Ave Subway and Regional Connections...  ...           -0.053536
31  AITAH for not letting construction crew use my...  ...            0.090375
37          Found a job thanks to this LinkedIn trick  ...           -0.168938
44                          CMV: All love is platonic  ...           -0.028085
57  These are the Democrats who just voted to keep...  ...            0.200360
59  Ranveer Singh endorsing the famously called "p...  ...            0.127804

[10 rows x 4 columns]